In [2]:
# Simple GenAI App using Langchain using Gemini

import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

# Used for Langsmith Tracking
os.environ["LANGCHAIN_GEMINI_API_KEY"] = os.getenv("LANGCHAIN_GEMINI_API_KEY")

# Used for Langchain Tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"

os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [3]:
# Data Ingestion- From website we scrape the data

from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.langchain.com/oss/python/langchain/messages")
loader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
documents = loader.load()
documents

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/messages', 'title': 'Messages - Docs by LangChain', 'language': 'en'}, page_content='Messages - Docs by LangChainSkip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationCore componentsMessagesDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityOn this pageBasic usageText promptsMessage promptsDictionary formatMessage typesSystem messageHuman messageText contentMessage metadataAI messageTool callsToken usageStrea

In [5]:
# Load data -> Docs -> Divide the docs into Text Chunks -> Embed the text into Vectors -> Store in vectorStoreDB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
docs = text_splitter.split_documents(documents)

docs

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/messages', 'title': 'Messages - Docs by LangChain', 'language': 'en'}, page_content='Messages - Docs by LangChainSkip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationCore componentsMessagesDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareAdvanced usageGuardrailsRuntimeContext engineeringModel'),
 Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/messages', 'title': 'Messages - Docs by LangChain', 'language': 'en'}, page_content='Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityOn t

In [6]:
from langchain_community.embeddings import OllamaEmbeddings

embeddings = (
    OllamaEmbeddings(model="nomic-embed-text:latest")
)
embeddings

/var/folders/dq/6104w49n4d96rlv9b113xvs40000gn/T/ipykernel_83219/859059093.py:4: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  OllamaEmbeddings(model="nomic-embed-text:latest")


OllamaEmbeddings(base_url='http://localhost:11434', model='nomic-embed-text:latest', embed_instruction='passage: ', query_instruction='query: ', mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None, show_progress=False, headers=None, model_kwargs=None)

In [7]:
from langchain_community.vectorstores import FAISS

vectorstoreDB = FAISS.from_documents(docs, embeddings)

In [8]:
vectorstoreDB

In [9]:
query = "Are Messages fundamental unit?"

result = vectorstoreDB.similarity_search(query)
print(result[0].page_content)

pageMessages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.


In [10]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

# Used for Langsmith Tracking
os.environ["LANGCHAIN_GEMINI_API_KEY"] = os.getenv("LANGCHAIN_GEMINI_API_KEY")

# Used for Langchain Tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"

os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")
print(llm)

profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True} google_api_key=SecretStr('**********') model='gemini-3-flash-preview' temperature=1.0 client=<google.genai.client.Client object at 0x1176593d0> default_metadata=() model_kwargs={}


In [ ]:
# Retrieval Chain, Document Chain -> to get context-aware response from LLMs
# Chains have been moved to the langchain_classic library

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
        Answer the following question based only on the provided context: 
            <context>
                {context}
            </context> 
    """
)

document_chain = create_stuff_documents_chain(llm, prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n        Answer the following question based only on the provided context: \n            <context>\n                {context}\n            </context> \n    '), additional_kwargs={})])
| ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_messa

In [14]:
from langchain_core.documents import Document

document_chain.invoke(
    {
        "input": "Messages are the fundamental unit of context",
        "context": [
            Document(
                page_content="""Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM. 
                                Messages are objects that contain:
                                Role - Identifies the message type (e.g. system, user)
                                Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
                                Metadata - Optional fields such as response information, message IDs, and token usage
                                LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called."""
            )
        ]
    }
)

'Based on the provided context, **Messages** are the fundamental unit of context for models in LangChain. They represent the input and output of models and carry the state of a conversation.\n\nMessages are objects that consist of:\n*   **Role:** Identifies the type of message (e.g., system, user).\n*   **Content:** The actual content, such as text, images, audio, or documents.\n*   **Metadata:** Optional fields including response information, message IDs, and token usage.\n\nLangChain uses a standard message type to ensure consistent behavior across all different model providers.'

In [18]:
retriever = vectorstoreDB.as_retriever()

from langchain_classic.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever, document_chain)

retrieval_chain


RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x130203e60>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n        Answer the following question based only on the provided context: \n            <context>\n                {context}\n            </context> \n   

In [ ]:
# Get the response from the LLM

response = retrieval_chain.invoke({
    "input": "Messages are the fundamental unit of context"
})

response['answer']

'Based on the context provided, **pageMessages** are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.'

In [20]:
response

{'input': 'Messages are the fundamental unit of context',
 'context': [Document(id='6c78b5f2-9405-4bb5-9fa9-58e5c07ef338', metadata={'source': 'https://docs.langchain.com/oss/python/langchain/messages', 'title': 'Messages - Docs by LangChain', 'language': 'en'}, page_content='pageMessages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.'),
  Document(id='d7998776-ec92-4b57-96bc-8a2cdaa306e0', metadata={'source': 'https://docs.langchain.com/oss/python/langchain/messages', 'title': 'Messages - Docs by LangChain', 'language': 'en'}, page_content='data.\u200bidstringUnique identifier for this content block (either generated by the provider or by LangChain).\u200bmime_typestringAudio MIME type (e.g., audio/mpeg, audio/wav). Required for base64 data.VideoContentBlockPurpose: Video data\u200btypestringrequiredAlways "vi